# 03 · Structured outputs with pydantic

After this notebook you can turn a pydantic model into a guaranteed JSON shape from the model (Chat Completions `.parse()` and Responses `.parse()`), handle optional fields, enums, nested lists and refusals under strict mode, repair invalid output with a feedback loop, and extract a batch of messy texts into a clean DataFrame, with the values checked, not just the shape.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · OpenAI API basics](01_openai_api_basics.ipynb), [type hints and pydantic](../01_python_basics/13_type_hints_and_pydantic.ipynb)

## Why this matters in interviews

- "How do you get reliable JSON out of a model?" is one of the most common practical questions. The ranked answer (constrained decoding / strict schemas > tool calling > validate-and-repair > prompt-only) is what they want.
- Every extraction, classification or routing feature ends in a data structure, and malformed output is where production incidents come from.
- The senior-level point: *schema-valid is not the same as correct*. Validate values against your own data too.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pr07` | How do you get reliable JSON out of a model? |
| `in20` | What is structured output / constrained decoding, and how does it work? |
| `ag35` | What is the difference between structured output and tool calling? |
| `tl20` | What does Pydantic do for an LLM application? |
| `sd17` | Design a system to extract structured data from unstructured documents at scale. |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import json, re
from enum import Enum
from typing import Literal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openai
from pydantic import BaseModel, Field, ValidationError, field_validator
from openai.types.chat import ParsedChatCompletionMessage

print("provider:", PROVIDER, "| model:", MODEL, "| openai", openai.__version__)

## 1. Why: "please reply in JSON" is not a contract

**In plain English:** if you only *ask* for JSON, you get JSON most of the time. At a million requests, "most of the time" is thousands of failures a day. Here are eight replies of the kinds models really produce when prompt-only JSON is requested. We run them through three levels of checking. No model is needed for this part.

In [ ]:
class Person(BaseModel):
    name: str
    age: int

RAW_REPLIES = {
    "clean":            '{"name": "Ana", "age": 31}',
    "markdown fence":   '```json\n{"name": "Ana", "age": 31}\n```',
    "chatty preamble":  'Sure! Here is the JSON:\n{"name": "Ana", "age": 31}',
    "trailing comma":   '{"name": "Ana", "age": 31,}',
    "python quotes":    "{'name': 'Ana', 'age': 31}",
    "wrong type":       '{"name": "Ana", "age": "thirty-one"}',
    "truncated":        '{"name": "Ana", "age": 31, "notes": "met at the',
    "wrong keys":       '{"full_name": "Ana", "years": 31}',
}

def naive(raw):
    return json.loads(raw)

def defensive(raw):
    """Strip code fences and chatter: parse the outermost {...} block."""
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip())
    start, end = raw.find("{"), raw.rfind("}")
    return json.loads(raw[start:end + 1])

def ok(fn, raw):
    try:
        fn(raw); return True
    except (ValueError, ValidationError):          # json.JSONDecodeError is a ValueError
        return False

table = pd.DataFrame({name: {"json.loads": ok(naive, raw),
                             "defensive parse": ok(defensive, raw),
                             "defensive + pydantic": ok(lambda r: Person.model_validate(defensive(r)), raw)}
                      for name, raw in RAW_REPLIES.items()}).T
print(table)
print("\nsuccess counts:", table.sum().to_dict())

In [ ]:
counts = table.sum()
fig, ax = plt.subplots(figsize=(8, 3.4))
bars = ax.bar(counts.index, counts.values, color=["#adb5bd", "#74c0fc", "#1c7ed6"])
ax.bar_label(bars, labels=[f"{v}/8" for v in counts.values])
ax.set_ylim(0, 9)
ax.set_ylabel("replies accepted")
ax.set_title("Parsing is not validating: only 3 of 8 replies are actually usable")
plt.show()
assert counts["json.loads"] == 3 and counts["defensive parse"] == 5 and counts["defensive + pydantic"] == 3

Read the table carefully: `json.loads` and pydantic each accept three replies, **but not the same three**. `json.loads` "succeeds" on *wrong type* and *wrong keys*, so it gives you garbage that looks fine. The defensive parser rescues the fence and the preamble. Only the pydantic check tells you which replies you can actually use. And no amount of parsing fixes the truncated one. (Never reach for `eval()` on the Python-style one: it executes whatever the model wrote.)

The real fix is to stop asking and start **constraining**: the provider masks every token that would break your schema during decoding, so the output is valid *by construction*.

## 2. Three ways to get structure

| | JSON mode | Structured outputs (strict JSON schema) | Tool calling (strict function) |
|---|---|---|---|
| How | `response_format={"type": "json_object"}` | `response_format={"type": "json_schema", ...}` or `.parse(response_format=Model)` | `tools=[...]` with `"strict": true` |
| Guarantees | syntactically valid JSON | JSON matching **your** schema: every key, type, enum | arguments matching the tool's schema |
| Does *not* guarantee | the keys or types you wanted | the *values* are true | that the model calls the tool at all (use `tool_choice`) |
| Where the JSON appears | `message.content` | `message.content`, parsed into `message.parsed` | `message.tool_calls[i].function.arguments` |
| Precondition | the word "JSON" in the messages | schema with `additionalProperties: false` and every field `required` | same, when `strict: true` |
| Use it for | old models, servers without schema support | extraction, classification, final answers | choosing an **action**, mid-loop ([notebook 04](04_function_calling.ipynb)) |

Structured output and tool calling use the same machinery. The difference is intent: with structured output you are *done*; with a tool call you are mid-loop and must execute something.

## 3. JSON mode: valid syntax, no promise about shape

In [ ]:
refund_text = "Order ORD10492 arrived damaged; please refund 34.50."
r = client.chat.completions.create(
    model=MODEL, response_format={"type": "json_object"},
    messages=[{"role": "user", "content": f"Return JSON with keys order_id, amount and reason for: {refund_text}"}])
reply = r.choices[0].message.content
print("JSON mode reply:", reply[:120])
data = json.loads(reply)                    # always parses...
print("keys:", list(data))

class Refund(BaseModel):
    order_id: str
    amount: float
    reason: Literal["damaged", "late", "duplicate", "other"]

try:
    print("valid Refund:", Refund.model_validate(data))
except ValidationError as e:
    print(f"...but does not validate: {e.error_count()} errors ->", [err["loc"][0] for err in e.errors()])

Offline, the stand-in's JSON-mode reply is `{"answer": ...}`: perfectly valid JSON with none of the keys you asked for. That is exactly the failure JSON mode permits. A real model that sees the key names in the prompt *usually* complies. "Usually" is the problem.

## 4. A pydantic model is a JSON schema

**In plain English:** you describe the shape once, as a Python class. Pydantic turns it into a JSON Schema (the model's contract) and later checks the reply against the same class.

In [ ]:
schema = Refund.model_json_schema()
print(json.dumps(schema, indent=1))

To be accepted in **strict** mode, OpenAI requires two things the default pydantic schema doesn't give you: `"additionalProperties": false` on every object, and **every** property listed in `required`. The SDK's `.parse()` adds both for you, recursively. You can see the converted schema through the public helper `openai.pydantic_function_tool` (it does the same conversion for tools):

In [ ]:
class LineItem(BaseModel):
    product: str
    quantity: int
    unit_price: float

class Order(BaseModel):
    order_id: str
    items: list[LineItem]                       # nested list of objects
    gift_note: str | None = None                # optional

strict = openai.pydantic_function_tool(Order)["function"]["parameters"]
print("pydantic 'required'      :", Order.model_json_schema()["required"])
print("strict 'required'        :", strict["required"])
print("strict additionalProps   :", strict["additionalProperties"], "| nested LineItem:",
      strict["$defs"]["LineItem"]["additionalProperties"])
print("gift_note in strict mode :", strict["properties"]["gift_note"]["anyOf"])
assert set(strict["required"]) == {"order_id", "items", "gift_note"}

Notice what happened to the optional field: `gift_note` is now **required but nullable** (`anyOf: [string, null]`). In strict mode "optional" means *the model must send the key, and may send `null`*. Declare optional fields as `X | None` and have your code treat `None` as "not stated".

## 5. `.parse()`: schema in, typed object out

In [ ]:
ticket_text = "Order ORD10492 arrived damaged; please refund 34.50."
r = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "system", "content": "Extract the refund request."},
              {"role": "user", "content": ticket_text}],
    response_format=Refund,
)
msg = r.choices[0].message
refund = msg.parsed
print(type(refund).__name__, "->", refund)
print(f"refund.amount + 1 = {refund.amount + 1}   (a real float, not a string)")
print("raw JSON the model sent:", msg.content)
assert isinstance(refund, Refund) and refund.order_id == "ORD10492" and refund.reason == "damaged"

The Responses API has the same helper: `text_format=` instead of `response_format=`, and `output_parsed` instead of `choices[0].message.parsed`.

In [ ]:
rr = client.responses.parse(
    model=MODEL,
    input=[{"role": "system", "content": "Extract the refund request."},
           {"role": "user", "content": ticket_text}],
    text_format=Refund,
)
print(type(rr.output_parsed).__name__, "->", rr.output_parsed)
assert rr.output_parsed == refund

## 6. Enums, nested lists, optional fields

`Literal[...]` and `Enum` both become a JSON-schema `enum`, so the model *cannot* invent a sixth category. Nested models become `$defs` references. Lists of objects are how you extract line items, entities or action items.

In [ ]:
class Priority(str, Enum):
    low = "low"
    normal = "normal"
    high = "high"

class ShipRequest(Order):
    priority: Priority

order_text = "Please ship order ORD77012 with high priority: 2 x Blue Mug at 9.50 each and 1 x Teapot at 24.00."
r = client.chat.completions.parse(model=MODEL, response_format=ShipRequest, messages=[
    {"role": "system", "content": "Extract the order. Use null for fields the text does not state."},
    {"role": "user", "content": order_text}])
order = r.choices[0].message.parsed
print("priority :", order.priority, "| is an Enum member:", isinstance(order.priority, Priority))
for item in order.items:
    print(f"  {item.quantity} x {item.product:<10} @ {item.unit_price:6.2f}")
print("gift_note:", order.gift_note)
print("total    :", sum(i.quantity * i.unit_price for i in order.items))
assert order.priority is Priority.high and all(isinstance(i, LineItem) for i in order.items)

A real model returns both line items (total 43.0) and `gift_note=None`, because the text mentions no gift note. The offline stand-in fills schemas mechanically: one list item, and a placeholder instead of `null`. Keep that in mind for section 10, because a model that fills a field the text never mentioned is exactly the error that value-checking has to catch.

## 7. The strict-schema 400

Hand-write a schema, forget `additionalProperties: false`, and a strict request is rejected before any tokens are generated:

In [ ]:
person_schema = {"type": "object",
                 "properties": {"name": {"type": "string"}, "age": {"type": "integer"}},
                 "required": ["name", "age"]}                       # no additionalProperties!

def ask_person(schema):
    return client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": "Ana is 31 years old."}],
        response_format={"type": "json_schema", "json_schema": {"name": "person", "strict": True, "schema": schema}})

try:
    ask_person(person_schema)
    print("accepted (this server does not enforce strict schemas)")
except openai.BadRequestError as e:
    print("400:", e.body.get("message") if isinstance(e.body, dict) else e)

fixed = ask_person({**person_schema, "additionalProperties": False})
print("fixed schema ->", fixed.choices[0].message.content)
assert Person.model_validate_json(fixed.choices[0].message.content).age == 31

Other strict-mode rules worth knowing: the root must be an object (wrap a list inside an object), there are limits on nesting depth and total properties, and some JSON-schema keywords are unsupported or limited. Check the provider docs when a schema gets exotic. Letting the SDK generate the schema from pydantic avoids nearly all of this.

## 8. Refusals and truncation

A strict schema can't force the model to produce harmful content. If it declines, you get a **refusal** instead of JSON: `message.refusal` holds the text and `parsed` is `None`. Your code needs that branch. Since a safe prompt never triggers one, the second message below is constructed by hand in the exact shape the SDK returns:

In [ ]:
def handle(message):
    if message.refusal:
        return {"status": "refused", "detail": message.refusal}
    if message.parsed is None:
        return {"status": "empty", "detail": None}
    return {"status": "ok", "detail": message.parsed.model_dump()}

refused = ParsedChatCompletionMessage[Refund](role="assistant", content=None, parsed=None,
                                             refusal="I'm sorry, I can't help with that request.")
for m in [msg, refused]:
    print(handle(m))
assert handle(refused)["status"] == "refused" and handle(msg)["status"] == "ok"

Truncation is the other one. If `finish_reason == "length"` (your `max_completion_tokens` cut the JSON off), `.parse()` raises `openai.LengthFinishReasonError` instead of handing you half an object. A filtered output raises `openai.ContentFilterFinishReasonError`. Catch both. The fix for truncation is a bigger cap or a smaller output, never "parse what arrived".

## 9. Validate values, retry with the validator's own words

**In plain English:** JSON Schema describes *shape*. Business rules ("order ids look like ORD + 5 digits", "refunds over $500 need a human") live in pydantic validators. When one fails, send the **exact error message** back to the model and ask it to fix that field. Cap the attempts: an uncapped repair loop is a money pump.

In [ ]:
class CheckedRefund(Refund):
    @field_validator("order_id")
    @classmethod
    def order_id_format(cls, v):
        if not re.fullmatch(r"ORD\d{5}", v):
            raise ValueError("order_id must be 'ORD' followed by exactly 5 digits")
        return v

    @field_validator("amount")
    @classmethod
    def amount_limit(cls, v):
        if not 0 < v <= 500:
            raise ValueError("amount must be > 0 and <= 500; larger refunds go to a human")
        return v

class ExtractionFailed(Exception):
    pass

def extract_with_feedback(text, call_model, max_attempts=3):
    """call_model(messages) -> raw JSON string. Returns (object, attempts_used)."""
    messages = [{"role": "system", "content": "Extract the refund request as JSON."},
                {"role": "user", "content": text}]
    for attempt in range(1, max_attempts + 1):
        raw = call_model(messages)
        try:
            return CheckedRefund.model_validate_json(raw), attempt
        except ValidationError as e:
            problems = "; ".join(f"{err['loc'][0]}: {err['msg']}" for err in e.errors())
            messages += [{"role": "assistant", "content": raw},
                         {"role": "user", "content": f"Your JSON failed validation: {problems}. "
                                                     "Return the corrected JSON only."}]
    raise ExtractionFailed(f"gave up after {max_attempts} attempts: {problems}")

Test the loop first with a **scripted fake model**: a list of canned replies. That's how you unit-test retry logic deterministically and for free. The first reply has the classic mistake (the id without its prefix), and the second is the corrected one.

In [ ]:
class ScriptedModel:
    """Replays canned replies and records what it was sent."""
    def __init__(self, replies):
        self.replies, self.received = list(replies), []
    def __call__(self, messages):
        self.received.append(messages[-1]["content"])
        return self.replies.pop(0)

fake = ScriptedModel(['{"order_id": "10492", "amount": 34.5, "reason": "damaged"}',
                      '{"order_id": "ORD10492", "amount": 34.5, "reason": "damaged"}'])
fixed_refund, attempts = extract_with_feedback("Order 10492 arrived damaged; refund 34.50.", fake)
print(f"attempts: {attempts} -> {fixed_refund}")
print("feedback the model saw on attempt 2:\n ", fake.received[1])
assert attempts == 2 and fixed_refund.order_id == "ORD10492"

stubborn = ScriptedModel(['{"order_id": "10492", "amount": 900, "reason": "damaged"}'] * 3)
try:
    extract_with_feedback("...", stubborn)
except ExtractionFailed as e:
    print("\ncapped:", e)

Now the same loop with the real client, using a strict schema built from the model. Validators are Python-side only, so they don't appear in the schema:

In [ ]:
def strict_format(model_cls):
    """Strict response_format for a *flat* model (the SDK's .parse() handles nesting for you)."""
    schema = {**model_cls.model_json_schema(), "additionalProperties": False}
    return {"type": "json_schema", "json_schema": {"name": model_cls.__name__, "strict": True, "schema": schema}}

def call_llm(messages):
    r = client.chat.completions.create(model=MODEL, messages=messages, response_format=strict_format(CheckedRefund))
    return r.choices[0].message.content

real_refund, attempts = extract_with_feedback("Order ORD10492 arrived damaged; please refund 34.50.", call_llm)
print(f"attempts: {attempts} -> {real_refund}")
assert real_refund.amount == 34.5

In production, log every validation failure along with the field and the prompt version. A rising repair rate on one field is a prompt (or schema-description) bug you can fix, and the repair loop is hiding it.

## 10. Batch extraction into a DataFrame, with values checked

Eight short support messages. Extract each one, collect the results in a DataFrame, and add a **grounding check**: every extracted string must actually appear in the source text. It is cheap, deterministic, and it catches invented values that are perfectly schema-valid.

In [ ]:
TEXTS = [
    "Hi, Maria Lopez here (maria@example.com). Order ORD10492 arrived broken and I want a refund. This is urgent!",
    "Tom Becker: where is my shipping update for ORD20931? It has been two weeks.",
    "I can't get into my account since the password reset. - Mamitha Nair",
    "Leo Park (leo.park@example.org) asks whether you sell gift cards.",
    "Ana Silva was charged twice for ORD33310 and needs a refund urgently.",
    "Sam Ortiz says the shipping status for ORD41177 shows delivered, but nothing arrived.",
    "Jo Kim wants the account email changed to jo.kim@example.com today.",
    "Kai Wong (kai@example.net) was billed for cancelled order ORD55821 and wants a refund urgently.",
]

class SupportTicket(BaseModel):
    customer: str = Field(description="the customer's full name")
    email: str | None = Field(description="email address if the text gives one, else null")
    order_id: str | None = Field(description="order id like ORD12345 if the text gives one, else null")
    category: Literal["other", "refund", "shipping", "account"]
    urgent: bool

SYSTEM = "Extract the support ticket fields. Use null for anything the text does not state. Never guess."
rows = []
for text in TEXTS:
    r = client.chat.completions.parse(model=MODEL, response_format=SupportTicket, messages=[
        {"role": "system", "content": SYSTEM}, {"role": "user", "content": text}])
    rows.append(r.choices[0].message.parsed.model_dump())
df = pd.DataFrame(rows)
print(df.to_string())

In [ ]:
CHECKED = ["customer", "email", "order_id"]

def grounding(value, source):
    if value is None:
        return "null"
    return "in text" if str(value).lower() in source.lower() else "NOT in text"

status = pd.DataFrame({f: [grounding(v, t) for v, t in zip(df[f], TEXTS)] for f in CHECKED})
df["ungrounded"] = [", ".join(f for f in CHECKED if status.loc[i, f] == "NOT in text") for i in range(len(df))]
print(df[["customer", "email", "order_id", "category", "ungrounded"]].to_string())
print("\nrows needing review:", int((df["ungrounded"] != "").sum()), "of", len(df))
assert (status["customer"] == "in text").all()

In [ ]:
kinds = ["in text", "null", "NOT in text"]
colors = {"in text": "#2f9e44", "null": "#adb5bd", "NOT in text": "#e03131"}
counts = pd.DataFrame({f: status[f].value_counts() for f in CHECKED}).reindex(kinds).fillna(0).T
fig, ax = plt.subplots(figsize=(8, 3.2))
left = np.zeros(len(CHECKED))
for k in kinds:
    ax.barh(CHECKED, counts[k], left=left, color=colors[k], label=k)
    left += counts[k].values
ax.invert_yaxis()
ax.set_xlim(0, 10.8)
ax.set_xticks(range(9))
ax.set_xlabel("number of the 8 messages")
ax.set_title("Grounding check per field: red = a value the source never mentions")
ax.legend(loc="center right", fontsize=8)
plt.show()

How to read this depends on the provider. Offline, the stand-in fills missing emails and order ids with placeholders (`user@example.com`, `12345`), and the check flags every one of them in red. That is exactly the "schema-valid, confidently invented" failure a real model can also produce under pressure. A good real model returns `null` (grey) for them, and nothing is red.

Either way, the pattern is the production design (`sd17`): constrained decoding for shape, validators for business rules, grounding and cross-field checks for truth, and anything flagged goes to human review. The business metric is the **straight-through rate**, the share of documents that need no human touch. Note that 95% per-field accuracy on a 20-field document can still send many documents to review.

## Try it yourself

**1. Add a cross-field rule.** Give `SupportTicket` a `model_validator(mode="after")` that rejects `category == "refund"` with `order_id is None` ("a refund needs an order id"). Re-validate the 8 extracted rows (no new model calls) and count the failures, then check that a hand-made refund row with no order id is rejected.

In [ ]:
# Solution — try it yourself first, then run this
from pydantic import model_validator

class StrictTicket(SupportTicket):
    @model_validator(mode="after")
    def refund_needs_order(self):
        if self.category == "refund" and self.order_id is None:
            raise ValueError("a refund needs an order id")
        return self

failures = []
for row in rows:
    try:
        StrictTicket.model_validate(row)
    except ValidationError as e:
        failures.append((row["customer"], e.errors()[0]["msg"]))
print(f"{len(failures)} of {len(rows)} extracted rows fail the cross-field rule")   # every refund text names an order
try:
    StrictTicket.model_validate({"customer": "Zoe Hart", "email": None, "order_id": None,
                                 "category": "refund", "urgent": False})
except ValidationError as e:
    print("hand-made row rejected:", e.errors()[0]["msg"])
assert len(failures) == 0

**2. Extract a list from one text.** Write a model `ActionItems` with `items: list[ActionItem]`, where `ActionItem` has `owner: str` and `task: str`, and extract it from: *"Mamitha will draft the spec. Tom will book the room."* Print each item. (A real model returns two items. The stand-in returns one, since it fills schemas mechanically.)

In [ ]:
# Solution — try it yourself first, then run this
class ActionItem(BaseModel):
    owner: str
    task: str

class ActionItems(BaseModel):
    items: list[ActionItem]

r = client.chat.completions.parse(model=MODEL, response_format=ActionItems, messages=[
    {"role": "system", "content": "List every action item with its owner."},
    {"role": "user", "content": "Mamitha will draft the spec. Tom will book the room."}])
for item in r.choices[0].message.parsed.items:
    print(f"- {item.owner}: {item.task}")
assert len(r.choices[0].message.parsed.items) >= 1

**3. Make the repair loop smarter.** A validation error on a *format* you can fix deterministically shouldn't cost a model call. Add a `field_validator("order_id", mode="before")` that turns `"10492"` or `"ord-10492"` into `"ORD10492"` before the strict check runs, and show that the scripted bad reply now passes on attempt 1.

In [ ]:
# Solution — try it yourself first, then run this
class NormalisedRefund(CheckedRefund):
    @field_validator("order_id", mode="before")
    @classmethod
    def normalise(cls, v):
        digits = re.sub(r"\D", "", str(v))
        return f"ORD{digits}" if len(digits) == 5 else v

bad_reply = '{"order_id": "ord-10492", "amount": 34.5, "reason": "damaged"}'
print(NormalisedRefund.model_validate_json(bad_reply))
assert NormalisedRefund.model_validate_json(bad_reply).order_id == "ORD10492"

## Say it in an interview

- **Reliable JSON, ranked (30 s):** "Constrain, don't ask. First choice is strict structured outputs (a JSON schema, often generated from a pydantic model), where the provider masks invalid tokens so the shape is guaranteed. For actions I use strict tool calling. When neither is available: parse defensively, validate with pydantic, and re-prompt with the validator's exact error, with the retries capped. Prompt-only JSON is the last resort."
- **Strict mode rules:** `additionalProperties: false` on every object; every property in `required`; optional = `X | None` (the key is present, the value may be null); the root is an object. The SDK's `.parse()` does the conversion. A hand-written schema that misses one of these is a 400.
- **Always branch on:** `message.refusal` (parsed is None), `LengthFinishReasonError` (truncated), `ContentFilterFinishReasonError`.
- **Pydantic's three jobs:** one class gives you the schema for the model, validation of the reply, and your API's request/response types. Validators encode business rules that JSON Schema can't express.
- **Trap:** "Schema-valid means correct?" No. A perfectly shaped object can contain an invented email or order id. Check values against the source and your own systems, and send flagged items to review. Constrained decoding also can't make the *content* right, and a schema the model finds unnatural can hurt quality.
- **Structured output vs tool calling:** same machinery. Structured output is the final answer; a tool call is an intermediate action your code executes, and the loop continues.

## Next

- [04 · Function calling](04_function_calling.ipynb): the same schemas, used to pick actions
- [LangChain structured output and tools](../13_langchain/03_structured_output_and_tools.ipynb) · [Tool design and function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb)
- [Type hints and pydantic](../01_python_basics/13_type_hints_and_pydantic.ipynb) · [Practical AI-engineer Python](../02_python_problems/04_practical_ai_engineer_python.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)